In [1]:
from delta.tables import DeltaTable
import pyspark.sql.functions as F

StatementMeta(, 60f06969-47b6-4a17-a9d1-1ee00be55484, 27, Finished, Available, Finished, False)

In [8]:
%run /nb_common_utils

StatementMeta(, 60f06969-47b6-4a17-a9d1-1ee00be55484, 31, Finished, Available, Finished, True)

In [9]:
# load_type = "FULL"
# run_id = 1

StatementMeta(, 60f06969-47b6-4a17-a9d1-1ee00be55484, 8, Finished, Available, Finished, False)

In [4]:
dim_table  = "gold.dim_customer"
stage_table = "stage.stg_new_customers"

StatementMeta(, 60f06969-47b6-4a17-a9d1-1ee00be55484, 3, Finished, Available, Finished, False)

In [11]:
ctx = log_start(
    spark,
    run_id    = run_id,
    file_name = "stg_new_customers",
    entity    = "customers",
    load_type = load_type
)


StatementMeta(, 60f06969-47b6-4a17-a9d1-1ee00be55484, 34, Finished, Available, Finished, False)

In [ ]:

df_stage = (
        spark.table(stage_table)
             .select(
                 F.col("customer_id"),
                 F.col("customer_name"),
                 F.col("city"),
                 F.col("segment"),
                 F.col("change_type"),
             )
    )

try:


    df_stage.cache()
    rows_read = df_stage.count()
    print(f"Rows read from stage: {rows_read}")


    # ADD AUDIT COLUMNS
    df_insert = (
        df_stage
            .filter(F.col("change_type").isin("INSERT", "UPSERT"))
            .drop("change_type")
            .withColumn("created_at", F.current_timestamp())
            .withColumn("updated_at", F.current_timestamp())
    )

    df_update = (
        df_stage
            .filter(F.col("change_type") == "UPDATE")
            .drop("change_type")
            .withColumn("updated_at", F.current_timestamp())
    )

    insert_count = df_insert.count()
    update_count = df_update.count()

 
    # FULL LOAD
    if load_type == "FULL":

        print("Running FULL LOAD — dim_customer...")

        (
            df_stage
                .drop("change_type")
                .withColumn("created_at", F.current_timestamp())
                .withColumn("updated_at", F.current_timestamp())
                .write
                .mode("overwrite")
                .format("delta")
                .saveAsTable(dim_table)
        )

        print(f"FULL LOAD completed: {rows_read} rows")

        log_end(spark, ctx,
                status        = "SUCCESS",
                rows_read     = rows_read,
                rows_inserted = rows_read)

  
    # INCREMENTAL LOAD

    elif load_type == "INCREMENTAL":

        print("Running INCREMENTAL LOAD — dim_customer (SCD Type 1)...")

        if not spark.catalog.tableExists(dim_table):

          
            print("dim_customer not found -> Initial load")

            (
                df_stage
                    .drop("change_type")
                    .withColumn("created_at", F.current_timestamp())
                    .withColumn("updated_at", F.current_timestamp())
                    .write
                    .mode("overwrite")
                    .format("delta")
                    .saveAsTable(dim_table)
            )

            print(f"Initial load completed: {rows_read} rows")

            log_end(spark, ctx,
                    status        = "SUCCESS",
                    rows_read     = rows_read,
                    rows_inserted = rows_read)

        else:

            delta_dim = DeltaTable.forName(spark, dim_table)

            # ── INSERT new customers ─────────────
            if insert_count > 0:

                (
                    df_insert.write
                        .mode("append")
                        .format("delta")
                        .saveAsTable(dim_table)
                )

                print(f"Inserted: {insert_count} new customers")

            else:
                print("No new customers to insert")

            # ── UPDATE existing customers ────────
            # SCD Type 1: 
            if update_count > 0:

                (
                    delta_dim.alias("t")
                    .merge(
                        df_update.alias("s"),
                        "t.customer_id = s.customer_id"
                    )
                    .whenMatchedUpdate(set={
                        "customer_name": "s.customer_name",
                        "city":          "s.city",
                        "segment":       "s.segment",
                        "updated_at":    "s.updated_at",
                    })
                    .execute()
                )

                print(f"Updated: {update_count} existing customers")

            else:
                print("No customers to update")

            if insert_count == 0 and update_count == 0:
                print("No changes detected — dim_customer unchanged")

            log_end(spark, ctx,
                    status        = "SUCCESS",
                    rows_read     = rows_read,
                    rows_inserted = insert_count,
                    rows_updated  = update_count)

    else:
        raise Exception(f"Unsupported load_type: {load_type}")

except Exception as e:
    log_end(spark, ctx, status="FAILED", error_message=str(e))
    raise

finally:
    df_stage.unpersist()


StatementMeta(, 60f06969-47b6-4a17-a9d1-1ee00be55484, 35, Finished, Available, Finished, False)

Rows read from stage: 0
Running FULL LOAD — dim_customer...


FULL LOAD completed: 0 rows
